# Final Project CRSP Setup
**Business 35137 — Winter 2026**

In [1]:
import pandas as pd
import numpy as np
import wrds
import os

DATA_DIR = '../Data/'
intermediate_dir = os.path.join(DATA_DIR, 'Intermediate')

### Part 1 Data Gathering: Monthly CRSP and Compustat

In [5]:
db = wrds.Connection(wrds_username=os.environ.get('WRDS_USERNAME'))  # set WRDS_USERNAME env var

# --- CRSP Monthly Returns (from 1950) ---
crsp = db.raw_sql("""
    SELECT a.permno, a.date, a.ret, a.prc, a.shrout, a.cfacpr,
           b.shrcd, b.exchcd, b.siccd, b.ticker, b.comnam
    FROM crsp_a_stock.msf AS a
    LEFT JOIN crsp_a_stock.msenames AS b
        ON a.permno = b.permno
        AND a.date BETWEEN b.namedt AND b.nameendt
    WHERE a.date >= '1950-01-01'
        AND b.shrcd IN (10, 11)       -- common shares only
""", date_cols=['date'])

# Compute market cap
crsp['mktcap'] = abs(crsp['prc']) * crsp['shrout']  # in thousands

# --- Compustat Annual Fundamentals (from 1950) ---
comp = db.raw_sql("""
    SELECT gvkey, datadate, tic, cusip, conm, sich, naicsh,
           at, sale, csho, prcc_f, ceq
    FROM comp.funda
    WHERE datadate >= '1950-01-01'
        AND indfmt = 'INDL'
        AND datafmt = 'STD'
        AND popsrc = 'D'
        AND consol = 'C'
""", date_cols=['datadate'])

# --- Compustat Company Info (address + country + industry) ---
comp_info = db.raw_sql("""
    SELECT gvkey, loc, sic, naics, state, city, addzip
    FROM comp.company
""")

comp = comp.merge(comp_info, on='gvkey', how='left')

# --- CCM Linking Table (PERMNO <-> GVKEY) ---
ccm = db.raw_sql("""
    SELECT gvkey, lpermno AS permno, linkdt, linkenddt, linktype, linkprim
    FROM crsp_a_ccm.ccmxpf_lnkhist
    WHERE linktype IN ('LU', 'LC')
        AND linkprim IN ('P', 'C')
""")

# --- S&P 500 constituents (historical) ---
sp500 = db.raw_sql("""
    SELECT permno, start AS start_dt, ending AS end_dt
    FROM crsp_a_indexes.dsp500list
""")

db.close()

Loading library list...
Done


In [6]:
# S&P 500 constituents (already fetched above)
sp500['permno'] = sp500['permno'].astype(int)
sp500['start_dt'] = pd.to_datetime(sp500['start_dt'])
sp500['end_dt'] = pd.to_datetime(sp500['end_dt'])


def flag_index_fast(crsp_df, index_df):
    """Flag index membership via an interval join."""
    merged = crsp_df[['permno', 'date']].merge(index_df, on='permno', how='inner')
    hits = merged.loc[
        (merged['date'] >= merged['start_dt']) & (merged['date'] <= merged['end_dt'])
    ][['permno', 'date']].drop_duplicates()
    return crsp_df.set_index(['permno', 'date']).index.isin(
        hits.set_index(['permno', 'date']).index
    )


# S&P 500 flag (numpy bool array, no NAs)
in_sp = np.array(flag_index_fast(crsp, sp500))

# Russell 2000 proxy: non-S&P stocks ranked 1001-3000 by market cap each month
crsp['_rank'] = crsp.groupby('date')['mktcap'].rank(ascending=False, method='first')
in_r2 = (~in_sp) & (crsp['_rank'].between(1001, 3000).fillna(False).values)
crsp.drop(columns='_rank', inplace=True)

# Single indicator: 'S&P 500', 'Russell 2000', or ''
crsp['index'] = np.where(in_sp, 'S&P 500',
                np.where(in_r2, 'Russell 2000', ''))

print(crsp['index'].value_counts(dropna=False))

# ── Atmospheric Industry Indicator ────────────────────────────────────────
sic = pd.to_numeric(crsp['siccd'], errors='coerce')

conditions = [
    sic.between(100, 999) | sic.between(2000, 2099),            # Agriculture & Food
    sic.between(4800, 4899) | sic.between(4900, 4999),           # Utilities & Telecom
    (sic.between(4500, 4599) | sic.between(5800, 5899)           # Leisure & Tourism
     | sic.between(7000, 7099) | sic.between(7900, 7999)),
]
labels = ['Agriculture & Food', 'Utilities & Telecom', 'Leisure & Tourism']

crsp['atmos_sector'] = np.select(conditions, labels, default='')

print('\n', crsp.loc[crsp['atmos_sector'] != '', 'atmos_sector'].value_counts())

index
                1877183
Russell 2000    1375713
S&P 500          398276
Name: count, dtype: int64

 atmos_sector
Utilities & Telecom    208558
Leisure & Tourism      110284
Agriculture & Food     103646
Name: count, dtype: int64


In [7]:
# ── 1. Prepare the CCM link table ─────────────────────────────────────────
ccm['permno'] = ccm['permno'].astype(int)
ccm['linkdt'] = pd.to_datetime(ccm['linkdt'])
ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt'].fillna('2099-12-31'))

# ── 2. Merge Compustat with CCM to get PERMNO ────────────────────────────
comp_ccm = comp.merge(ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']],
                      on='gvkey', how='inner')

# Keep only rows where the fiscal year-end falls within the valid link window
comp_ccm = comp_ccm[
    (comp_ccm['datadate'] >= comp_ccm['linkdt']) &
    (comp_ccm['datadate'] <= comp_ccm['linkenddt'])
].drop(columns=['linkdt', 'linkenddt'])

# ── 3. Create a merge key: Compustat fiscal year → next-year June ────────
# Standard Fama-French timing: annual data known by June of year t+1
comp_ccm['year'] = comp_ccm['datadate'].dt.year
comp_ccm['merge_month'] = pd.to_datetime(
    (comp_ccm['year'] + 1).astype(str) + '-07-01'   # available from July t+1
)

# For each permno-year, keep only the latest fiscal year-end
comp_ccm = comp_ccm.sort_values('datadate').drop_duplicates(
    subset=['permno', 'year'], keep='last'
)

# ── 4. Prepare CRSP merge key (year-month for matching) ──────────────────
crsp['merge_month'] = crsp['date'].dt.to_period('M').dt.to_timestamp()

# Assign each CRSP month to the Compustat vintage it should use:
#   months July t  – June t+1  →  fiscal data from calendar year t
crsp['comp_year'] = np.where(
    crsp['date'].dt.month >= 7,
    crsp['date'].dt.year,       # Jul-Dec → same year
    crsp['date'].dt.year - 1    # Jan-Jun → prior year
)

# ── 5. Merge CRSP with Compustat ─────────────────────────────────────────
comp_cols = ['permno', 'year', 'gvkey', 'datadate', 'tic', 'cusip', 'conm',
             'sich', 'naicsh', 'at', 'sale', 'csho', 'prcc_f', 'ceq',
             'state', 'city', 'addzip', 'loc']

master_crsp = crsp.merge(
    comp_ccm[comp_cols].rename(columns={'year': 'comp_year'}),
    on=['permno', 'comp_year'],
    how='left'
)

# Clean up helper columns
master_crsp.drop(columns=['merge_month', 'comp_year'], inplace=True)

# ── 6. Quick diagnostics ─────────────────────────────────────────────────
n_total = len(master_crsp)
n_matched = master_crsp['gvkey'].notna().sum()
print(f"Total CRSP-month obs : {n_total:,.0f}")
print(f"Matched to Compustat : {n_matched:,.0f}  ({n_matched/n_total:.1%})")
print(f"Unmatched            : {n_total - n_matched:,.0f}")
print(f"\nDate range: {master_crsp['date'].min():%Y-%m} – {master_crsp['date'].max():%Y-%m}")
print(f"Unique PERMNOs       : {master_crsp['permno'].nunique():,.0f}")
print(f"Unique GVKEYs        : {master_crsp['gvkey'].nunique():,.0f}")

# ── Missingness among MATCHED rows ────────────────────────────────────────
matched = master_crsp[master_crsp['gvkey'].notna()]
check_cols = ['state', 'city', 'addzip', 'county', 'loc',
              'at', 'sale', 'ceq', 'sich', 'naicsh']
print("\n── Column fill rates among matched rows ──")
for col in check_cols:
    if col in matched.columns:
        n_filled = matched[col].notna().sum()
        # Also check for empty strings
        if matched[col].dtype == object:
            n_filled = (matched[col].notna() & (matched[col] != '')).sum()
        pct = n_filled / len(matched) * 100
        print(f"  {col:10s}: {n_filled:>12,.0f} / {len(matched):,.0f}  ({pct:5.1f}%)")

print(f"\nIndex membership:\n{master_crsp['index'].value_counts(dropna=False)}")
print(f"\nAtmospheric sectors:\n{master_crsp.loc[master_crsp['atmos_sector'] != '', 'atmos_sector'].value_counts()}")
master_crsp.sort_values("date").head()

Total CRSP-month obs : 3,651,172
Matched to Compustat : 3,156,504  (86.5%)
Unmatched            : 494,668

Date range: 1950-01 – 2024-12
Unique PERMNOs       : 26,454
Unique GVKEYs        : 22,365

── Column fill rates among matched rows ──
  state     :    3,075,792 / 3,156,504  ( 97.4%)
  city      :    3,043,647 / 3,156,504  ( 96.4%)
  addzip    :    3,039,249 / 3,156,504  ( 96.3%)
  loc       :    3,156,504 / 3,156,504  (100.0%)
  at        :    3,109,471 / 3,156,504  ( 98.5%)
  sale      :    3,098,130 / 3,156,504  ( 98.2%)
  ceq       :    3,007,398 / 3,156,504  ( 95.3%)
  sich      :    2,006,570 / 3,156,504  ( 63.6%)
  naicsh    :    2,214,922 / 3,156,504  ( 70.2%)

Index membership:
index
                1877183
Russell 2000    1375713
S&P 500          398276
Name: count, dtype: int64

Atmospheric sectors:
atmos_sector
Utilities & Telecom    208558
Leisure & Tourism      110284
Agriculture & Food     103646
Name: count, dtype: int64


,permno,date,ret,prc,shrout,cfacpr,shrcd,exchcd,siccd,ticker,...,naicsh,at,sale,csho,prcc_f,ceq,state,city,addzip,loc
441055,14795,1950-01-31,-0.027778,35.0,2421.0,23.205015,11,1,3560,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
794327,21370,1950-01-31,0.034935,29.625,1179.0,24.544374,11,1,4910,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
647613,18569,1950-01-31,0.002212,55.625,1951.0,27.568457,10,1,3010,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
813863,21784,1950-01-31,0.170455,12.875,423.0,1.21,10,1,3660,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
598173,17742,1950-01-31,-0.019231,12.75,785.0,5.061341,10,1,3710,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
#master_crsp.to_parquet(os.path.join(intermediate_dir, 'master_crsp_merged.parquet'))